# MixUpLLaVA-Video-R1 · 总控制台

**可组合 GRPO 实验台**主入口（Phase 6）。汇总 Phase 1–5：默认档 C1、模块库、选择器、训练期评估。

| 约定 | 说明 |
|------|------|
| 默认策略 | **B + CPPO**（除非显式关闭） |
| 评测默认 | **训练期**指标；四基准总评估可选（~600GB） |
| 开训 | **按需**；本台先把配置与报告链路跑通即可验收 |

**固定流程**：环境 → 调参 → 选策略 → 基线计划/训/评 → 优化计划/训/评 → 对比报告

Phase1 参考（同档）：A2 steps/s **0.008** · C2 **0.011**（+37.5%）；详见 `docs/PHASE1_REPORT.md`。


## A. 环境与路径（先跑）


In [ ]:
import sys
from pathlib import Path

TRY_MOUNT = False  # Colab 可改为 True
if TRY_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/MixUpLLaVA-video-r1")
MIXUP_REPO = PROJECT_DIR / "repo" / "MixUpLLaVA-Video-R1"
TINY_REPO = PROJECT_DIR / "repo" / "TinyLLaVA-Video-R1"

if not (MIXUP_REPO / "mixup").is_dir():
    MIXUP_REPO = Path.cwd()
    if not (MIXUP_REPO / "mixup").is_dir():
        MIXUP_REPO = Path.cwd().parent
    PROJECT_DIR = MIXUP_REPO
    TINY_REPO = PROJECT_DIR / "repo" / "TinyLLaVA-Video-R1"

sys.path.insert(0, str(MIXUP_REPO))
print("PROJECT_DIR", PROJECT_DIR)
print("MIXUP_REPO ", MIXUP_REPO)
print("TINY_REPO  ", TINY_REPO, "exists=", TINY_REPO.is_dir())


## B. 总览


In [ ]:
from mixup.presets import PRESET_ALIASES, list_presets
from mixup.registry import list_modules
from mixup.trainer_mixup import ensure_modules_loaded
from mixup.console_reports import PHASE1_REFERENCE

ensure_modules_loaded()
print("Presets:", PRESET_ALIASES)
print("Modules:", [m.name for m in list_modules()])
print("Phase1 ref:", PHASE1_REFERENCE)


## C. 训练超参（Phase 2 · 默认 C1）

用 `tier_overrides` 覆盖；与 Phase1 并表时勿改核心档位。


In [ ]:
from mixup.config_loader import load_tier_config, default_c1_path, apply_config_to_notebook

TIER_OVERRIDES = {
    # "max_steps": 20,
    # "num_generations": 4,
}
tier = load_tier_config(default_c1_path(MIXUP_REPO), overrides=TIER_OVERRIDES or None)
apply_config_to_notebook(tier, globals())
print(f"tier={tier.get('tier')} g={NUM_GENERATIONS} steps={MAX_STEPS} frames={NUM_FRAMES}")


## D. 策略选择（Phase 4）

先 **基线计划**（默认 `m1` = B+CPPO），再 **优化方案计划**。
`APPLY_PATCH=True` 需 Drive 上 TinyLLaVA REPO。


In [ ]:
from mixup.training_entry import prepare_training, describe_launch

APPLY_PATCH = False
DRY_RUN = False

BASE_PRESET = "m1"
BASE_MIXUP = None
BASELINE_PLAN = prepare_training(
    preset=BASE_PRESET,
    mixup=BASE_MIXUP,
    project_dir=PROJECT_DIR if PROJECT_DIR.is_dir() else None,
    mixup_repo=MIXUP_REPO,
    tinyllava_repo=TINY_REPO if TINY_REPO.is_dir() else None,
    apply_patch=APPLY_PATCH,
    dry_run=DRY_RUN,
    tag="baseline",
    notes="console baseline M1",
    tier_overrides=TIER_OVERRIDES or None,
)
print("=== BASELINE ===")
print(BASELINE_PLAN.summary())
print(describe_launch(BASELINE_PLAN))


In [ ]:
CAND_PRESET = "m1"
CAND_MIXUP = {
    "ngrpo": True,
    # "gfpo": True, "gfpo_top_k": 4,
    # "mo_grpo": True,
}
CANDIDATE_PLAN = prepare_training(
    preset=CAND_PRESET,
    mixup=CAND_MIXUP,
    project_dir=PROJECT_DIR if PROJECT_DIR.is_dir() else None,
    mixup_repo=MIXUP_REPO,
    tinyllava_repo=TINY_REPO if TINY_REPO.is_dir() else None,
    apply_patch=APPLY_PATCH,
    dry_run=DRY_RUN,
    tag="cand",
    notes="console candidate",
    tier_overrides=TIER_OVERRIDES or None,
)
print("=== CANDIDATE ===")
print(CANDIDATE_PLAN.summary())
print("dropped defaults:", CANDIDATE_PLAN.dropped_project_defaults)


## E. 基线：训练（按需）→ 训练期评估 → 基线报告

开训复用 Phase1 `run_training` → `BASELINE_PLAN.output_dir`。未开训也可生成报告骨架。


In [ ]:
from mixup.console_reports import write_baseline_report, refresh_training_eval
from pathlib import Path

base_dir = Path(BASELINE_PLAN.output_dir)
if base_dir.is_dir() and (
    (base_dir / "trainer_state.json").exists()
    or list(base_dir.glob("checkpoint-*/trainer_state.json"))
):
    refresh_training_eval(base_dir)
    print("training_eval refreshed")

baseline_report = write_baseline_report(BASELINE_PLAN)
print("Wrote", baseline_report)


## F. 优化方案：训练（按需）→ 训练期评估


In [ ]:
from mixup.console_reports import refresh_training_eval
from pathlib import Path

cand_dir = Path(CANDIDATE_PLAN.output_dir)
if cand_dir.is_dir() and (
    (cand_dir / "trainer_state.json").exists()
    or list(cand_dir.glob("checkpoint-*/trainer_state.json"))
):
    refresh_training_eval(cand_dir, baseline_dir=BASELINE_PLAN.output_dir)
    print("candidate training_eval refreshed")
else:
    print("No trainer_state yet — compare report will be a stub until you train.")


## G. 对比报告 + Phase1 历史对照


In [ ]:
from mixup.console_reports import write_compare_report, PHASE1_REFERENCE

compare_path = write_compare_report(BASELINE_PLAN, CANDIDATE_PLAN)
print("Wrote", compare_path)
print("Phase1 A2/C2 reference:")
for k, v in PHASE1_REFERENCE.items():
    print(f"  {k}: {v}")


## H. （可选）四基准总评估

默认跳过。见 `docs/PHASE5_BENCHMARKS.md`。


In [ ]:
from mixup.eval_benchmarks import build_benchmark_plan, run_benchmark

EVAL_ROOT = PROJECT_DIR / "data" / "eval"
RUN_FULL_BENCH = False

if TINY_REPO.is_dir():
    plan = build_benchmark_plan(
        eval_root=EVAL_ROOT,
        model_path=CANDIDATE_PLAN.output_dir,
        tinyllava_repo=TINY_REPO,
    )
    print(plan.to_dict()["statuses"])
    if RUN_FULL_BENCH:
        print(run_benchmark(
            "videomme",
            mixup_repo=MIXUP_REPO,
            tinyllava_repo=TINY_REPO,
            model_path=CANDIDATE_PLAN.output_dir,
            eval_root=EVAL_ROOT,
            dry_run=False,
            confirm_large_download=True,
        ))
else:
    print("TINY_REPO missing — skip benchmark plan")


## 完成检查

- [ ] 基线 / 方案 `run_id` 与 `mixup_config.json` 已落盘
- [ ] `baseline_report.md` / `compare_vs_baseline.md` 已生成
- [ ] （可选）已开训并刷新训练期指标
- [ ] （可选）四基准仅在有数据时运行

文档：`docs/CONSOLE_GUIDE.md` · `docs/MIXUP_MODULES.md` · `docs/PHASE5_BENCHMARKS.md`
